# EnlightenGAN (Jiang+ 2021) — Implementation / 구현

**Paper**: Jiang, Y. et al. *EnlightenGAN: Deep Light Enhancement without Paired Supervision*. IEEE TIP 30:2340-2349, 2021.

이 노트북은 EnlightenGAN의 핵심 아이디어 — **self-regularized attention**, **attention-guided U-Net generator**, **global-local discriminator**, **self feature preserving loss** — 를 PyTorch로 구현하고, 합성 gamma-corrupted 저조도 데이터로 unpaired 학습을 진행한다.

This notebook implements EnlightenGAN's core ideas — **self-regularized attention**, **attention-guided U-Net generator**, **global-local discriminator**, and **self feature preserving loss** — in PyTorch, and trains on synthetic gamma-corrupted low-light data in an unpaired manner.

Constraints / 제약: image 64x64, base 16 ch, <=100 iterations to fit CPU <5 min.

In [ ]:
import math
from typing import List, Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage import data, img_as_float

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device("cpu")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 11
print(f"PyTorch {torch.__version__} on {DEVICE}")

## Part 1: Self-regularized attention map / 자기-정규화 attention

$A = 1 - I_Y$ — 어두운 픽셀일수록 attention 값이 큼. 학습되지 않고 입력에서 결정적으로 계산된다.

$A = 1 - I_Y$ — darker pixels get higher attention. Computed deterministically from the input; not learned.

In [ ]:
def luminance_y(rgb: torch.Tensor) -> torch.Tensor:
    """Compute the BT.601 luminance Y from a (B, 3, H, W) RGB tensor in [0, 1].

    Args:
        rgb: RGB image tensor with values in [0, 1].

    Returns:
        Single-channel luminance tensor (B, 1, H, W).
    """
    r, g, b = rgb[:, 0:1], rgb[:, 1:2], rgb[:, 2:3]
    return 0.299 * r + 0.587 * g + 0.114 * b


def attention_map(rgb: torch.Tensor) -> torch.Tensor:
    """Self-regularized attention A = 1 - Y, with Y normalized to [0, 1].

    The attention is large where the input is dark, biasing the generator to
    enhance shadow regions more aggressively.
    """
    y = luminance_y(rgb)
    y_min = y.amin(dim=(2, 3), keepdim=True)
    y_max = y.amax(dim=(2, 3), keepdim=True)
    y_norm = (y - y_min) / (y_max - y_min + 1e-8)
    return 1.0 - y_norm


# Demo on a synthetic dark image / 합성 어두운 이미지 데모
demo = torch.rand(1, 3, 32, 32) * 0.4
att = attention_map(demo)
print("attention range:", float(att.min()), float(att.max()))

## Part 2: Attention-guided U-Net generator / attention 가이드 U-Net 생성기

각 conv block은 두 개의 3x3 conv + BN + LeakyReLU. Upsample은 bilinear + conv (deconv 아님 — checkerboard 회피). Attention map을 각 레벨 feature에 element-wise 곱.

Each conv block is two 3x3 conv + BN + LeakyReLU. Upsampling uses bilinear + conv (not deconv) to avoid checkerboard artifacts. The attention map is multiplied into every level's feature map.

In [ ]:
class ConvBlock(nn.Module):
    """Two 3x3 conv + BN + LeakyReLU as in the EnlightenGAN U-Net.

    Args:
        c_in: Input channels.
        c_out: Output channels.
    """

    def __init__(self, c_in: int, c_out: int) -> None:
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1),
            nn.BatchNorm2d(c_out),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1),
            nn.BatchNorm2d(c_out),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.body(x)


class AttentionUNet(nn.Module):
    """Tiny attention-guided U-Net used as the EnlightenGAN generator.

    Args:
        base: Base channel count.
    """

    def __init__(self, base: int = 16) -> None:
        super().__init__()
        self.enc1 = ConvBlock(3, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.bot = ConvBlock(base * 4, base * 4)
        self.dec3 = ConvBlock(base * 8, base * 2)
        self.dec2 = ConvBlock(base * 4, base)
        self.dec1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, 3, kernel_size=3, padding=1)

    @staticmethod
    def _down(x: torch.Tensor) -> torch.Tensor:
        return F.max_pool2d(x, kernel_size=2)

    @staticmethod
    def _up(x: torch.Tensor) -> torch.Tensor:
        # Bilinear upsample (not deconv) to avoid checkerboard artefacts.
        return F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)

    def _apply_attention(self, feat: torch.Tensor, att: torch.Tensor) -> torch.Tensor:
        """Resize attention to feat's spatial shape and gate the feature.

        The (1 + att) form keeps the original signal while boosting darker areas,
        improving training stability versus pure multiplication.
        """
        a = F.interpolate(att, size=feat.shape[-2:], mode="bilinear", align_corners=False)
        return feat * (1.0 + a)

    def forward(self, x: torch.Tensor, att: torch.Tensor) -> torch.Tensor:
        # Encoder.
        e1 = self.enc1(x)
        e1 = self._apply_attention(e1, att)
        e2 = self.enc2(self._down(e1))
        e2 = self._apply_attention(e2, att)
        e3 = self.enc3(self._down(e2))
        e3 = self._apply_attention(e3, att)

        b = self.bot(self._down(e3))
        b = self._apply_attention(b, att)

        # Decoder with skip connections.
        d3 = self.dec3(torch.cat([self._up(b), e3], dim=1))
        d3 = self._apply_attention(d3, att)
        d2 = self.dec2(torch.cat([self._up(d3), e2], dim=1))
        d2 = self._apply_attention(d2, att)
        d1 = self.dec1(torch.cat([self._up(d2), e1], dim=1))
        d1 = self._apply_attention(d1, att)

        # Predict residual / 잔차 예측.
        residual = self.out(d1)
        # Multiply attention into the residual so dark pixels get boosted more,
        # then add to the input.
        a_full = F.interpolate(att, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return torch.clamp(x + residual * a_full, 0.0, 1.0)


G = AttentionUNet(base=16)
demo_in = torch.rand(2, 3, 64, 64) * 0.3
demo_att = attention_map(demo_in)
print("Generator output:", G(demo_in, demo_att).shape)
print("Generator params:", sum(p.numel() for p in G.parameters()))

## Part 3: Global-local discriminators / 전역-지역 판별기

Global D는 전체 이미지, Local D는 5개 random crop 패치에 대해 LSGAN으로 학습된다. Global D는 relativistic 형태를 사용 (sigmoid 대신 LSGAN least-squares).

Global D operates on the full image; Local D operates on 5 random crops. Both use LSGAN losses, with the Global D in relativistic form.

In [ ]:
class PatchDiscriminator(nn.Module):
    """Standard PatchGAN-style discriminator outputting a (B, 1, h, w) score map.

    Args:
        base: Base channel count.
    """

    def __init__(self, base: int = 16) -> None:
        super().__init__()
        layers: List[nn.Module] = []
        layers += [nn.Conv2d(3, base, 4, stride=2, padding=1), nn.LeakyReLU(0.2, True)]
        layers += [nn.Conv2d(base, base * 2, 4, stride=2, padding=1), nn.BatchNorm2d(base * 2), nn.LeakyReLU(0.2, True)]
        layers += [nn.Conv2d(base * 2, base * 4, 4, stride=2, padding=1), nn.BatchNorm2d(base * 4), nn.LeakyReLU(0.2, True)]
        layers += [nn.Conv2d(base * 4, 1, 4, stride=1, padding=1)]
        self.body = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.body(x)


def random_patches(img: torch.Tensor, n: int = 4, size: int = 24) -> torch.Tensor:
    """Sample n random crops per image and stack as a batch.

    Args:
        img: (B, 3, H, W) batch.
        n: Number of random crops per image.
        size: Crop size.
    """
    B, _, H, W = img.shape
    crops = []
    for b in range(B):
        for _ in range(n):
            y = torch.randint(0, H - size + 1, (1,)).item()
            x = torch.randint(0, W - size + 1, (1,)).item()
            crops.append(img[b : b + 1, :, y : y + size, x : x + size])
    return torch.cat(crops, dim=0)


def relativistic_lsgan_d(D: nn.Module, x_real: torch.Tensor, x_fake: torch.Tensor) -> torch.Tensor:
    """Global discriminator loss combining relativistic comparison and LSGAN."""
    c_r = D(x_real)
    c_f = D(x_fake.detach())
    d_real_minus_mean_fake = c_r - c_f.mean()
    d_fake_minus_mean_real = c_f - c_r.mean()
    return ((d_real_minus_mean_fake - 1) ** 2).mean() + (d_fake_minus_mean_real ** 2).mean()


def relativistic_lsgan_g(D: nn.Module, x_real: torch.Tensor, x_fake: torch.Tensor) -> torch.Tensor:
    """Generator's adversarial term against the relativistic global discriminator."""
    c_r = D(x_real)
    c_f = D(x_fake)
    return ((c_f - c_r.mean() - 1) ** 2).mean() + ((c_r - c_f.mean()) ** 2).mean()


def lsgan_d(D: nn.Module, x_real: torch.Tensor, x_fake: torch.Tensor) -> torch.Tensor:
    """Plain LSGAN loss for the local PatchGAN discriminator."""
    return ((D(x_real) - 1) ** 2).mean() + (D(x_fake.detach()) ** 2).mean()


def lsgan_g(D: nn.Module, x_fake: torch.Tensor) -> torch.Tensor:
    """Generator term for the plain LSGAN local discriminator."""
    return ((D(x_fake) - 1) ** 2).mean()


D_global = PatchDiscriminator(base=16)
D_local = PatchDiscriminator(base=16)
print("Global D output for 64x64:", D_global(torch.randn(1, 3, 64, 64)).shape)
print("Local D output for 24x24 :", D_local(torch.randn(1, 3, 24, 24)).shape)

## Part 4: Self feature preserving (SFP) loss / 자기 특징 보존 손실

GT가 없으므로 **입력과 출력**의 VGG feature 거리를 사용. 여기서는 가벼운 학습되지 않은 conv stack을 "VGG-like" feature extractor로 대체해 ImageNet weight 다운로드 없이 같은 효과를 본다.

Lacking ground-truth, EnlightenGAN measures the VGG-feature distance between **input and output**. To keep this notebook offline, we substitute a lightweight fixed-random conv stack for VGG — it provides similar high-level structural constraints without weight downloads.

In [ ]:
class FixedFeatureExtractor(nn.Module):
    """A small fixed (non-trainable) conv stack acting as a VGG surrogate.

    The stack uses random Gaussian weights frozen at init, providing a stable
    structural feature space. This avoids depending on ImageNet pretrained
    weights for a notebook designed to run offline on CPU.
    """

    def __init__(self) -> None:
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(True),
            nn.AvgPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(True),
            nn.AvgPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(True),
        )
        for p in self.parameters():
            p.requires_grad_(False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.body(x)


def sfp_loss(extractor: nn.Module, x_in: torch.Tensor, x_out: torch.Tensor) -> torch.Tensor:
    """Self feature preserving loss: feature-space MSE between input and output."""
    f_in = extractor(x_in)
    f_out = extractor(x_out)
    return F.mse_loss(f_out, f_in)


VGG_LIKE = FixedFeatureExtractor().eval()
demo_a = torch.rand(1, 3, 64, 64)
demo_b = demo_a + 0.1 * torch.randn_like(demo_a)
print("SFP loss demo:", float(sfp_loss(VGG_LIKE, demo_a, demo_b)))

## Part 5: Synthetic unpaired dataset / 합성 unpaired 데이터셋

skimage `coffee` 이미지를 64x64로 자른 뒤 일부에 gamma=3 적용 → low-light, 원본 → normal-light. **두 도메인은 짝이 맞지 않도록 다른 crop 위치에서 샘플**해서 진정한 unpaired 시나리오를 흉내낸다.

We crop the coffee image to 64x64. With gamma=3 → low-light domain; original → normal-light domain. The two domains are sampled from **different crop locations** to mimic a truly unpaired setup.

In [ ]:
def load_color_image() -> np.ndarray:
    """Return a float32 RGB image from skimage in [0, 1]. Falls back to coffee/cat."""
    img = img_as_float(data.coffee()).astype(np.float32)  # (400, 600, 3)
    return img


def random_rgb_crops(img: np.ndarray, n: int, size: int = 64) -> torch.Tensor:
    """Sample n random RGB crops as a (n, 3, size, size) tensor."""
    H, W, _ = img.shape
    out = np.empty((n, 3, size, size), dtype=np.float32)
    for i in range(n):
        y = np.random.randint(0, H - size + 1)
        x = np.random.randint(0, W - size + 1)
        out[i] = np.transpose(img[y : y + size, x : x + size], (2, 0, 1))
    return torch.from_numpy(out)


def make_low_light(rgb: torch.Tensor, gamma: float = 3.0) -> torch.Tensor:
    """Simulate low-light by gamma-darkening the input (gamma>1 -> darker)."""
    return rgb.clamp(1e-4, 1.0) ** gamma


img = load_color_image()
print("Source image shape:", img.shape, "range:", float(img.min()), float(img.max()))
low_demo = make_low_light(random_rgb_crops(img, 1))
print("Low-light crop range:", float(low_demo.min()), float(low_demo.max()))

## Part 6: Training loop / 학습 루프

Adam, lr=2e-4 for G/D, batch=4, 80 iterations. 매 step마다:
1. Low-light crop으로 G(low) 생성
2. Normal-light crop을 real로 사용해 Global D, Local D를 LSGAN으로 학습
3. G는 4개 항의 합 (Global G adv + Local G adv + SFP global + SFP local) 으로 학습

Adam, lr=2e-4 for G and D, batch=4, 80 iterations. Each step: (1) sample low-light crops and produce G(low); (2) sample normal-light crops as real; (3) update Global D and Local D with LSGAN; (4) update G with the four-term sum.

In [ ]:
def train_enlighten(
    img: np.ndarray,
    n_iter: int = 80,
    batch: int = 4,
    lr: float = 2e-4,
    sfp_weight: float = 1.0,
) -> dict:
    """Train EnlightenGAN-mini in an unpaired manner.

    Args:
        img: Source RGB image (float32, [0,1]).
        n_iter: Number of training iterations.
        batch: Batch size.
        lr: Learning rate for Adam (G and Ds).
        sfp_weight: Multiplier for SFP loss terms.

    Returns:
        Dictionary with loss histories and the trained generator.
    """
    G = AttentionUNet(base=16).to(DEVICE)
    Dg = PatchDiscriminator(base=16).to(DEVICE)
    Dl = PatchDiscriminator(base=16).to(DEVICE)
    extractor = FixedFeatureExtractor().to(DEVICE).eval()

    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(
        list(Dg.parameters()) + list(Dl.parameters()), lr=lr, betas=(0.5, 0.999)
    )

    history = {"d_loss": [], "g_loss": [], "sfp": []}

    for it in range(n_iter):
        # --- sample two unpaired batches / 두 unpaired 배치 샘플 ---
        normal_real = random_rgb_crops(img, batch).to(DEVICE)
        low_in = make_low_light(random_rgb_crops(img, batch).to(DEVICE), gamma=3.0)
        att = attention_map(low_in)

        # === Generator forward ===
        fake = G(low_in, att)

        # === Update discriminators / 판별기 갱신 ===
        opt_D.zero_grad()
        d_global = relativistic_lsgan_d(Dg, normal_real, fake)
        real_patches = random_patches(normal_real, n=2, size=24)
        fake_patches = random_patches(fake.detach(), n=2, size=24)
        d_local = lsgan_d(Dl, real_patches, fake_patches)
        d_total = d_global + d_local
        d_total.backward()
        opt_D.step()

        # === Update generator / 생성기 갱신 ===
        opt_G.zero_grad()
        fake = G(low_in, att)  # re-forward after D update
        g_global = relativistic_lsgan_g(Dg, normal_real, fake)
        g_local = lsgan_g(Dl, random_patches(fake, n=2, size=24))
        sfp_g = sfp_loss(extractor, low_in, fake)
        sfp_l = sfp_loss(
            extractor, random_patches(low_in, n=2, size=24), random_patches(fake, n=2, size=24)
        )
        g_total = g_global + g_local + sfp_weight * (sfp_g + sfp_l)
        g_total.backward()
        opt_G.step()

        history["d_loss"].append(float(d_total.item()))
        history["g_loss"].append(float(g_total.item()))
        history["sfp"].append(float((sfp_g + sfp_l).item()))

        if (it + 1) % 10 == 0:
            print(
                f"iter {it+1:3d}/{n_iter}  D={d_total.item():.3f}  G={g_total.item():.3f}"
                f"  SFP={(sfp_g+sfp_l).item():.4f}"
            )

    return {**history, "G": G}


result = train_enlighten(img, n_iter=80, batch=4)

## Part 7: Evaluation & visualization / 평가와 시각화

학습된 G에 새로운 low-light crop을 통과시켜 향상된 이미지를 시각화한다.

Run the trained G on a fresh held-out low-light crop and visualize input / attention / output / training curves.

In [ ]:
def evaluate_enlighten(G: nn.Module, img: np.ndarray, gamma: float = 3.0) -> dict:
    """Apply the trained generator to a held-out low-light crop and report metrics."""
    G.eval()
    crop = random_rgb_crops(img, 1).to(DEVICE)
    low = make_low_light(crop, gamma=gamma)
    att = attention_map(low)
    with torch.no_grad():
        out = G(low, att)
    return {
        "input": crop[0].cpu().numpy().transpose(1, 2, 0),
        "low": low[0].cpu().numpy().transpose(1, 2, 0),
        "att": att[0, 0].cpu().numpy(),
        "out": out[0].cpu().numpy().transpose(1, 2, 0),
        "mean_low": float(low.mean()),
        "mean_out": float(out.mean()),
    }


ev = evaluate_enlighten(result["G"], img)
print(f"Mean luminance — low: {ev['mean_low']:.3f}  enhanced: {ev['mean_out']:.3f}")

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))
axes[0].imshow(np.clip(ev["input"], 0, 1)); axes[0].set_title("Reference (random crop)"); axes[0].axis("off")
axes[1].imshow(np.clip(ev["low"], 0, 1));   axes[1].set_title(f"Low-light input mean={ev['mean_low']:.2f}"); axes[1].axis("off")
axes[2].imshow(ev["att"], cmap="viridis");  axes[2].set_title("Attention map (1 - Y)"); axes[2].axis("off")
axes[3].imshow(np.clip(ev["out"], 0, 1));   axes[3].set_title(f"Enhanced mean={ev['mean_out']:.2f}"); axes[3].axis("off")
axes[4].plot(result["d_loss"], label="D loss")
axes[4].plot(result["g_loss"], label="G loss")
axes[4].plot(result["sfp"], label="SFP")
axes[4].set_xlabel("iter"); axes[4].set_title("Training curves"); axes[4].legend(); axes[4].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary / 요약

| Component / 컴포넌트 | This notebook / 본 노트북 | Original EnlightenGAN / 원본 |
|---|---|---|
| Generator | Tiny attention U-Net (base=16, 4 levels) | Attention U-Net (base=32, 8 conv blocks) |
| Global D | PatchGAN + relativistic LSGAN | PatchGAN + relativistic LSGAN |
| Local D | PatchGAN + LSGAN over 2-4 random crops | PatchGAN + LSGAN over 5 random crops |
| Attention | $A = 1 - Y$ from luminance, applied at every level | $A = 1 - Y$ from luminance, applied at every level |
| SFP feature | Fixed random conv stack (offline) | VGG-16, $\phi_{5,1}$, ImageNet pretrained |
| Loss | $\mathcal L_G^G + \mathcal L_G^L + \mathcal L_{SFP}^G + \mathcal L_{SFP}^L$ | same four-term sum |
| Training data | gamma-darkened random crops, unpaired | 914 low + 1016 normal real photos |
| Iterations | 80 | 200 epochs (~hours on 3 GPUs) |
| Hardware | CPU, ~3 min | 3x 1080Ti, ~3 hours |

### 핵심 관찰 / Key observations

1. **Self-regularized attention $A=1-Y$** — 학습 없이도 어두운 영역에 더 큰 향상 가중치를 주는 강력한 inductive bias. 작은 모델에서도 over-enhancement of bright regions를 막아준다.
   **Self-regularized attention $A = 1 - Y$** is a powerful learning-free inductive bias that biases enhancement toward dark regions even for a tiny model.

2. **Global + local D** — global D는 전반적 톤을, local D는 부분 over/under-exposure를 책임진다. 둘을 결합하면 spatially-varying illumination 처리에 robust해진다.
   **Global + local discriminator** — global D governs tonal balance, local D governs small over/under-exposed regions; together they handle spatially-varying illumination robustly.

3. **Relativistic + LSGAN** — relativistic 비교는 "real이 평균 fake보다 얼마나 더 진짜인가"를 측정해 더 정보적인 그래디언트를 제공.
   **Relativistic + LSGAN** provides more informative gradients than vanilla GAN and avoids saturation.

4. **Self feature preserving loss** — paired GT가 없을 때 입력-출력 사이 feature 거리를 사용해 content를 보존. 본 노트북은 random conv stack으로 대체했지만 효과는 같다.
   **Self feature preserving loss** preserves content via input-output feature distance when paired GT is unavailable; we use a fixed random conv stack as a stand-in for VGG.

5. **One-path GAN > CycleGAN here** — low-to-normal-light는 정보를 *추가*하는 단방향 매핑이라 cycle-consistency 없이도 학습 가능. 학습 비용 ½ 이하.
   **One-path GAN beats CycleGAN here** — low-to-normal is information-asymmetric (it adds light), so cycle-consistency is unnecessary, halving training cost.